# Data Parameter Prepare

## import

In [14]:
import configparser
import torch
import numpy as np
import scipy.sparse as sp
import torch.utils.data as data
import os
import pickle

## data_param_prepare()

In [21]:
def data_param_prepare(config_file):
    
    # get config
    config = configparser.ConfigParser()
    config.read(config_file)
    
    params = {}
    
    embedding_dim = config.getint('Model', 'embedding_dim')
    params['embedding_dim'] = embedding_dim
    ii_neighbor_num = config.getint('Model', 'ii_neighbor_num')
    params['ii_neighbor_num'] = ii_neighbor_num
    model_save_path = config['Model']['model_save_path']
    params['model_save_path'] = model_save_path
    max_epoch = config.getint('Model', 'max_epoch')
    params['max_epoch'] = max_epoch
    
    params['enable_tensorboard'] = config.getboolean('Model', 'enable_tensorboard')
    
    initial_weight = config.getfloat('Model', 'initial_weight')
    params['initial_weight'] = initial_weight
    
    dataset = config['Training']['dataset']
    params['dataset'] = dataset
    train_file_path = config['Training']['train_file_path']
    gpu = config['Training']['gpu']
    params['gpu'] = gpu
    device = torch.device('cuda:'+params['gpu'] if torch.cuda.is_available() else "cpu")
    params['device'] = device
    lr = config.getfloat('Training', 'learning_rate')
    params['lr'] = lr
    batch_size = config.getint('Training', 'batch_size')
    params['batch_size'] = batch_size
    early_stop_epoch = config.getint('Training', 'early_stop_epoch')
    params['early_stop_epoch'] = early_stop_epoch
    w1 = config.getfloat('Training', 'w1')
    w2 = config.getfloat('Training', 'w2')
    w3 = config.getfloat('Training', 'w3')
    w4 = config.getfloat('Training', 'w4')
    params['w1'] = w1
    params['w2'] = w2
    params['w3'] = w3
    params['w4'] = w4
    negative_num = config.getint('Training', 'negative_num')
    negative_weight = config.getfloat('Training', 'negative_weight')
    params['negative_num'] = negative_num
    params['negative_weight'] = negative_weight
    
    gamma = config.getfloat('Training', 'gamma')
    params['gamma'] = gamma
    lambda_ = config.getfloat('Training', 'lambda') # labmda는 예약어라 변수명으로 사용 못함
    params['lambda'] = lambda_
    sampling_sift_pos = config.getboolean('Training', 'sampling_sift_pos') # 부정 샘플링(negative sampling) 과정에서 긍정 항목을 걸러낼지(sift) 여부를 결정하는 옵션
    params['sampling_sift_pos'] = sampling_sift_pos
    
    test_batch_size = config.getint('Testing', 'test_batch_size')
    params['test_batch_size'] = test_batch_size
    topk = config.getint('Testing', 'topk')
    params['topk'] = topk
    
    test_file_path = config['Testing']['test_file_path']
    
    # dataset processing
    train_data, test_data, train_mat, user_num, item_num, constraint_mat = load_data(train_file_path, test_file_path) # call load_data()
    train_loader = data.DataLoader(train_data, batch_size=batch_size, shuffle = True, num_workers=5)
    test_loader = data.DataLoader(list(range(user_num)), batch_size=test_batch_size, shuffle=False, num_workers=5)
    
    params['user_num'] = user_num
    params['item_num'] = item_num
    
    # mask matrix for testing to accelarate testing speed
    mask = torch.zeros(user_num, item_num)
    interacted_items = [[] for _ in range(user_num)]
    for (u, i) in train_data:
        mask[u][i] = -np.inf
        interacted_items[u].append(i)
        
    # test user-item interaction, which is ground truth
    test_ground_truth_list = [[] for _ in range(user_num)]
    for (u, i) in test_data:
        test_ground_truth_list[u].append(i)
        
    # Compute \Omega to extend UltraGCN to the item-item co-occurrence graph
    ii_cons_mat_path = './' + dataset + '_ii_constraint_mat'
    ii_neigh_mat_path = './' + dataset + '_ii_neighbor_mat'
    
    if os.path.exists(ii_cons_mat_path):
        ii_constraint_mat = pload(ii_cons_mat_path) # call pload()
        ii_neighbor_mat = pload(ii_neigh_mat_path)
    else:
        ii_neighbor_mat, ii_constraint_mat = get_ii_constraint_mat(train_mat, ii_neighbor_num) # call get_ii_constraint_mat()
        pstore(ii_neighbor_mat, ii_neigh_mat_path) # call pstore()
        pstore(ii_constraint_mat, ii_cons_mat_path)
    
    return params, constraint_mat, ii_constraint_mat, ii_neighbor_mat, train_loader, test_loader, mask, test_ground_truth_list, interacted_items
    

In [24]:
def load_data(train_file, test_file):
    trainUniqueUsers, trainItem, trainUser = [], [], []
    testUniqueUsers, testItem, testUser = [], [], []
    n_user, m_item = 0, 0
    trainDataSize, testDataSize = 0, 0
    with open(train_file, 'r') as f:
        for l in f.readlines():
            if len(l) > 0:
                l = l.strip('\n').split(' ')
                items = [int(i) for i in l[1:]]
                uid = int(l[0])
                trainUniqueUsers.append(uid)
                trainUser.extend([uid] * len(items))
                trainItem.extend(items)
                m_item = max(m_item, max(items))
                n_user = max(n_user, uid)
                trainDataSize += len(items)
    trainUniqueUsers = np.array(trainUniqueUsers)
    trainUser = np.array(trainUser)
    trainItem = np.array(trainItem)
    
    with open(test_file) as f:
        for l in f.readlines():
            if len(l) > 0:
                l = l.strip('\n').split(' ')
                try:
                    items = [int(i) for i in l[1:]]
                except:
                    items = []
                uid = int(l[0])
                testUniqueUsers.append(uid)
                testUser.extend([uid] * len(items))
                testItem.extend(items)
                try:
                    m_item = max(m_item, max(items))
                except:
                    m_item = m_item
                n_user = max(n_user, uid)
                testDataSize += len(items)
                
    train_data = []
    test_data = []
    
    n_user += 1
    m_item += 1
    
    for i in range(len(trainUser)):
        train_data.append([trainUser[i], trainItem[i]])
    for i in range(len(testUser)):
        test_data.append([testUser[i], testItem[i]])
    train_mat = sp.dok_matrix((n_user, m_item), dtype=np.float32)
    
    for x in train_data:
        train_mat[x[0], x[1]] = 1.0
    
    # construct degree matrix for graphmf
    
    items_D = np.sum(train_mat, axis = 0).reshape(-1)
    users_D = np.sum(train_mat, axis = 1).reshape(-1)
    
    beta_uD = (np.sqrt(users_D + 1) / users_D).reshape(-1, 1)
    beta_iD = (1 / np.sqrt(items_D + 1)).reshape(1, -1)
    
    constraint_mat = {"beta_uD": torch.from_numpy(beta_uD).reshape(-1), "beta_iD": torch.from_numpy(beta_iD).reshape(-1)}
    
    return train_data, test_data, train_mat, n_user, m_item, constraint_mat

In [18]:
def pload(path):
    with open(path, 'rb') as f:
        res = pickle.load(f)
    print('store object in path = {} ok'.format(path))
    return res

In [23]:
def get_ii_constraint_mat(train_mat, num_neighbors, ii_diagonal_zero = False):
    print('Computing \\Omega for the item-item graph... ')
    A = train_mat.T.dot(train_mat) # I * I
    n_items = A.shape[0]
    res_mat = torch.zeros((n_items, num_neighbors))
    res_sim_mat = torch.zeros((n_items, num_neighbors))
    if ii_diagonal_zero:
        A[range(n_items), range(n_items)] = 0
    items_D = np.sum(A, axis = 0).reshape(-1)
    users_D = np.sum(A, axis = 1).reshape(-1)
    
    beta_uD = (np.sqrt(users_D + 1) / users_D).reshape(-1, 1)
    beta_iD = (1 / np.sqrt(items_D + 1)).reshape(1, -1)
    all_ii_constraint_mat = torch.from_numpy(beta_uD.dot(beta_iD))
    for i in range(n_items):
        row = all_ii_constraint_mat[i] * torch.from_numpy(A.getrow(i).toarray()[0])
        row_sims, row_idxs = torch.topk(row, num_neighbors)
        res_mat[i] = row_idxs
        res_sim_mat[i] = row_sims
        if i % 15000 == 0:
            print('i-i constraint matrix {} ok'.format(i))
    
    print('Computation \\Omega OK!')
    return res_mat.long(), res_sim_mat.float()

In [27]:
def pstore(x, path):
    with open(path, 'wb') as f:
        pickle.dump(x, f)
    print('store object in path = {} ok'.format(path))

In [28]:
# .ini 파일 경로 지정
config_file = './config/ultragcn_yelp18_m1.ini'

# data_param_prepare 함수 호출
params, constraint_mat, ii_constraint_mat, ii_neighbor_mat, train_loader, test_loader, mask, test_ground_truth_list, interacted_items = data_param_prepare(config_file)

# 실험 결과 확인
# 1. 설정된 학습 파라미터 확인
print("Model Parameters:")
for param, value in params.items():
    print(f"{param}: {value}")

# 2. 제약 행렬(Constraint Matrices) 확인
print("\nConstraint Matrix:", constraint_mat)
print("Item-Item Constraint Matrix:", ii_constraint_mat)
print("Item-Item Neighbor Matrix:", ii_neighbor_mat)

# 3. DataLoader 확인 - 학습 데이터 샘플
print("\nTrain Data Samples:")
for batch in train_loader:
    print(batch)
    break  # 첫 번째 배치만 출력

# 4. 마스크 행렬과 사용자-아이템 상호작용 확인
print("\nMask Matrix (sample):")
print(mask[:5, :5])  # 상위 5x5 부분만 확인

print("\nTest Ground Truth List (sample):")
print(test_ground_truth_list[:5])

print("\nInteracted Items (sample):")
print(interacted_items[:5])


Computing \Omega for the item-item graph... 
i-i constraint matrix 0 ok
i-i constraint matrix 15000 ok
i-i constraint matrix 30000 ok
Computation \Omega OK!
store object in path = ./yelp2018_ii_neighbor_mat ok
store object in path = ./yelp2018_ii_constraint_mat ok
Model Parameters:
embedding_dim: 64
ii_neighbor_num: 10
model_save_path: ./ultragcn_yelp2018.pt
max_epoch: 2000
enable_tensorboard: True
initial_weight: 0.0001
dataset: yelp2018
gpu: 0
device: cuda:0
lr: 0.001
batch_size: 1024
early_stop_epoch: 15
w1: 1e-08
w2: 1.0
w3: 1e-08
w4: 1.0
negative_num: 800
negative_weight: 300.0
gamma: 0.0001
lambda: 0.0005
sampling_sift_pos: False
test_batch_size: 2048
topk: 20
user_num: 31668
item_num: 38048

Constraint Matrix: {'beta_uD': tensor([0.2422, 0.0432, 0.1222,  ..., 0.1923, 0.2130, 0.2496]), 'beta_iD': tensor([0.0416, 0.2887, 0.1400,  ..., 0.7071, 0.7071, 0.7071])}
Item-Item Constraint Matrix: tensor([[0.0156, 0.0040, 0.0032,  ..., 0.0031, 0.0030, 0.0029],
        [0.0268, 0.0116, 0.00